# Advanced packages - Processing output

Part of the **synthetic-valley advanced-packages** series. This notebook runs the shared advanced model in `models/` - whatever advanced packages the other notebooks have built into it - and plots the results with the calibration targets.

- **UZF** (`mf6-adv-uzf`) - recharge (RCH) -> Unsaturated Zone Flow
- **MAW** (`mf6-adv-maw`) - wells (WEL) -> Multi-Aquifer Well
- **LAK** (`mf6-adv-lak`) - high-K lake -> Lake package
- **SFR** (`mf6-adv-sfr`) - river (RIV) -> Streamflow Routing
- **MVR** (`mf6-adv-mvr`) - Water Mover: runoff and seepage to the lake and stream (requires UZF, LAK, SFR)
- **Processing output** (`mf6-adv-processing`) - run the model and plot results *(this notebook)*

**Run order.** If you run this notebook before building any advanced packages, `load_or_create_advanced_model` copies the calibrated base model into `models/` and this notebook runs and post-processes that base model. As you build UZF, MAW, LAK, SFR, and MVR with the other notebooks, re-running this notebook reflects whichever packages are present: the post-processing detects the SFR and LAK packages and switches the surface-water budget term and streamflow/lake plots accordingly.

## What this notebook covers

Run the model the rest of the series built, and judge it: compare simulated
heads with the calibration targets, look at drawdown and streamflow under
pumping, and see where the model is biased.

By the end of this notebook you will be able to:

- run the assembled advanced model and read its heads, budgets, and
  observations,
- compare simulated and observed heads and summarize the residuals,
- map drawdown and read streamflow depletion from the gauge, and
- tell a calibration problem from a model that is behaving as built.


## Imports and setup

Import FloPy and the helpers, then choose the temporal resolution (`sample_frequency`) and the model name.

In [ ]:
%matplotlib inline
import flopy
import matplotlib.pyplot as plt
import matplotlib.tri as tri
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator
from mf6_notebook_helpers import (
    label_map_axes,
    length_name,
    load_land_surface,
    load_or_create_advanced_model,
    load_temporal_data,
    load_truth,
    synthetic_valley_workspaces,
    to_display,
    unit,
)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

## Load and run the advanced model

Load the shared advanced model from `models/` (or create it from the base model on first use), run it, and read the grid dimensions. Then detect which advanced packages are present so the plots below adapt to the current state of the model.

In [ ]:
sim = load_or_create_advanced_model(sample_frequency, name)
gwf = sim.get_model(name)

success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

nlay, nrow, ncol = gwf.dis.nlay.array, gwf.dis.nrow.array, gwf.dis.ncol.array
shape2d = (nrow, ncol)
_, advanced_ws = synthetic_valley_workspaces(sample_frequency, name)

In [ ]:
# detect the advanced packages that have been built into the model; the plots
# below choose what to draw from these instead of hard-coded toggles
has_sfr = gwf.get_package("SFR-1") is not None
has_lak = gwf.get_package("LAK-1") is not None
print(f"SFR present: {has_sfr}, LAK present: {has_lak}")
print("packages:", gwf.get_package_list())

### Load the supporting data

Read the lake cells, the forcing, and the truth: the heads and observed series of the same valley simulated on a finer Voronoi grid, with its own hydraulic conductivity in each layer and the clay's vertical hydraulic conductivity. The truth plays the part of field measurements. Then compute the elapsed time at the end of the calibration period and at the end of the two prediction periods.

In [ ]:
lake = load_land_surface()["lake"].to_numpy() == 1
temporal_df = load_temporal_data(sample_frequency)
truth = load_truth()

In [ ]:
idx_end_calibration = 0
if sample_frequency == "monthly":
    idx_end_period2 = 120
    idx_end_period3 = 240
elif sample_frequency == "annual":
    idx_end_period2 = 10
    idx_end_period3 = 20
else:
    raise ValueError(f"invalid sample_frequency: '{sample_frequency}'")

start_date = pd.to_datetime("1962-01-01 00:00:00")

end_calibration = temporal_df.index[idx_end_calibration]
end_period_two = temporal_df.index[idx_end_period2]
end_period_three = temporal_df.index[idx_end_period3]

totim_end = [float((end_calibration - start_date).days)]
totim_end += [float((end_period_two - start_date).days)]
totim_end += [float((end_period_three - start_date).days)]
totim_end

## Plot the results

Visualize the simulated results and compare them with the calibration targets.

### Model properties

Maps of hydraulic conductivity, bottom elevation, and cell thickness by layer. Define `plot_layers()` once here — it draws a `(nlay, nrow, ncol)` array as a row of by-layer maps in the USGS map style (with labeled white contours when `levels` is given) — and reuse it for every by-layer map below. Lengths are shown in the display units (`MF6_TRAINING_UNITS`).

In [ ]:
def plot_layers(array, title, levels=None):
    """Map a (nlay, nrow, ncol) array layer by layer in a row of USGS-style map
    axes; overlay labeled white contours when `levels` is given."""
    with flopy.plot.styles.USGSMap():
        fig, axs = plt.subplots(1, nlay, figsize=(9, 3), sharey=True)
        fig.suptitle(title)
        for idx in range(nlay):
            ax = axs[idx]
            mm = flopy.plot.PlotMapView(
                model=gwf,
                layer=idx,
                ax=ax,
            )
            mm.plot_array(array)
            if levels is not None:
                cs = mm.contour_array(array, colors="white", levels=levels)
                plt.clabel(cs, inline=True, fontsize=8)
            ax.set_title(f"Layer {idx + 1}")
            ax.set_xticks([])
            ax.set_yticks([])


def active(array):
    """Mask the cells that are inactive, dry, or pass water straight through."""
    idomain = gwf.dis.idomain.array
    return np.ma.masked_where((idomain < 1) | (np.abs(array) >= 1.0e20), array)

In [ ]:
# the base model's lake cells, with their very high hydraulic conductivity, are left out
k = np.ma.masked_greater_equal(active(gwf.npf.k.array), 1.0e5)
plot_layers(
    to_display(k, "rate"), f"Hydraulic conductivity, in {unit('rate', plain=True)}"
)

In [ ]:
plot_layers(
    to_display(active(gwf.dis.botm.array)), f"Bottom elevation, in {length_name()}"
)

In [ ]:
plot_layers(
    to_display(active(gwf.modelgrid.cell_thickness)),
    f"Cell thickness, in {length_name()}",
)

**What to look for.** These three maps show the model's fixed properties by layer, before anything is simulated: horizontal hydraulic conductivity (how easily water moves; the base model's lake cells, if the Lake package has not been built yet, are left out), the bottom elevation of each cell, and the cell thickness. Layer 3 is the clay, which covers only the northern part of the valley; elsewhere its cells have no thickness and pass water straight between layers 2 and 4, so they are blank.

### Simulated heads and drawdown

A helper to retrieve heads at a given time (substituting lake stage in the lake cells when the LAK package is present), followed by the head and drawdown maps for each period.

In [ ]:
def get_heads(totim):
    """Heads at a time, with the lake level in the lake cells of layer 1 when the
    Lake package is present."""
    hds = gwf.output.head().get_data(totim=totim)
    if has_lak:
        stage = gwf.lak.output.stage().get_data(totim=totim)
        hds[0][lake] = stage.flatten()[0]
    return active(hds)

In [ ]:
# head contour levels in the display units, from the range of the calibration heads
levels = MaxNLocator(12).tick_values(
    *to_display(
        np.array([get_heads(totim_end[0]).min(), get_heads(totim_end[0]).max()])
    )
)
levels

#### Calibration

Simulated heads at the end of the calibration period, the steady state before pumping.

In [ ]:
totim = totim_end[0]
hds = get_heads(totim)

In [ ]:
plot_layers(to_display(hds), f"Heads - Calibration, in {length_name()}", levels=levels)

**What to look for.** These are the simulated heads at the end of the calibration period, contoured by layer. Groundwater flows from the valley sides toward the stream and south, and the heads in the two aquifers are close where the clay is absent. The residuals below compare them with the truth at the observation wells.

**Inspecting the budget.** The cell below lists the flow terms MODFLOW saved in the cell-by-cell budget file and then picks the right label for the surface-water exchange - `"sfr"` when the SFR package is active, `"riv"` otherwise - so the post-processing below works no matter which packages are present.

In [ ]:
# list the flow terms saved in the cell-by-cell budget file, then choose the
# label for the surface-water exchange: "sfr" when the SFR package is active,
# "riv" otherwise
budget = gwf.output.budget()
record_names = [
    r.strip() for r in np.array(budget.get_unique_record_names()).astype(str)
]
print("Available budget terms:", record_names)

riv_text = "sfr" if has_sfr else "riv"
v = budget.get_data(text=riv_text, totim=totim)[0]["q"]
print(f"\nUsing '{riv_text}' for the surface-water exchange term")
print(f"River infiltration everywhere positive: {np.all(v > 0)}\n{v}")

##### Calculate the residuals

Compare simulated and true heads at the 17 observation wells, at the water table (layer 1) and in the lower aquifer (layer 4 or 5), at the end of the calibration period. The model records the heads at the wells as observations, and the truth holds the same observations from the Voronoi-grid valley.

In [ ]:
# simulated heads at the observation wells, and the true heads at the same wells
calibration = {}
for where in ("wt", "aq"):
    simulated = pd.read_csv(advanced_ws / f"{name}.gwf.{where}.csv", index_col=0)
    simulated = simulated.loc[totim_end[0]]
    observed = np.array(
        [float(truth[f"obs_{well.lower()}"].isel(time=0)) for well in simulated.index]
    )
    calibration[where] = pd.DataFrame(
        {"simulated": simulated.to_numpy(), "observed": observed}, index=simulated.index
    )
    calibration[where]["residual"] = (
        calibration[where]["simulated"] - calibration[where]["observed"]
    )
calibration["wt"]

In [ ]:
# the structured cell of each observation well, from its name (e.g. WT01-I2-J17)
obs_rc_locs = [
    (int(well.split("-")[1][1:]), int(well.split("-")[2][1:]))
    for well in calibration["wt"].index
]

In [ ]:
resid_wt = to_display(calibration["wt"]["residual"].to_numpy())
resid_aq = to_display(calibration["aq"]["residual"].to_numpy())
resid_gb = np.concatenate((resid_wt, resid_aq))

In [ ]:
for label, resid in (
    ("Water table", resid_wt),
    ("Lower aquifer", resid_aq),
    ("All wells", resid_gb),
):
    print(
        f"{label}: mean error {resid.mean():.2f} {unit(plain=True)}, "
        f"root mean square error {np.sqrt((resid**2).mean()):.2f} {unit(plain=True)}"
    )

##### Plot the residuals

Interpolate and contour the head residuals to check for spatial bias in the calibration.

In [ ]:
xy = [
    (float(gwf.modelgrid.xcellcenters[i, j]), float(gwf.modelgrid.ycellcenters[i, j]))
    for i, j in obs_rc_locs
]
x, y = np.array(xy)[:, 0], np.array(xy)[:, 1]
grid_x, grid_y = np.meshgrid(gwf.modelgrid.xycenters[0], gwf.modelgrid.xycenters[1])

# Linearly interpolate the residuals (x, y) onto the grid
triang = tri.Triangulation(x, y)
interpolator = tri.LinearTriInterpolator(triang, resid_wt)
grid_resid_wt = interpolator(grid_x, grid_y)
interpolator = tri.LinearTriInterpolator(triang, resid_aq)
grid_resid_aq = interpolator(grid_x, grid_y)

# contour levels in the display units, the same on both sides of zero
resid_levels = MaxNLocator(10, symmetric=True).tick_values(
    -np.abs(resid_gb).max(), np.abs(resid_gb).max()
)

In [ ]:
with flopy.plot.styles.USGSMap():
    fig, axs = plt.subplots(1, 2, figsize=(8, 5), sharey=True)
    fig.suptitle(f"Head residuals, simulated minus true, in {length_name()}")

    ax = axs[0]
    mm = flopy.plot.PlotMapView(
        model=gwf,
        ax=ax,
        extent=gwf.modelgrid.extent,
    )
    mm.plot_array(np.ma.masked_equal(lake.astype(int), 0), cmap="Blues_r")
    mm.plot_grid(lw=0.5, color="0.5")
    ax.scatter(x, y, s=3, c="black")
    for i, txt in enumerate(resid_wt):
        ax.annotate(f"{txt:.2f}", (x[i], y[i]))
    cs = ax.contour(
        grid_x,
        grid_y,
        grid_resid_wt,
        levels=resid_levels,
        linewidths=0.75,
        colors="red",
    )
    plt.clabel(cs, inline=True, fontsize=8)
    ax.set_title("Water Table")
    label_map_axes(ax)

    ax = axs[1]
    mm = flopy.plot.PlotMapView(
        model=gwf,
        ax=ax,
        extent=gwf.modelgrid.extent,
    )
    mm.plot_grid(lw=0.5, color="0.5")
    ax.scatter(x, y, s=3, c="black")
    for i, txt in enumerate(resid_aq):
        ax.annotate(f"{txt:.2f}", (x[i], y[i]), clip_on=False)
    cs = ax.contour(
        grid_x,
        grid_y,
        grid_resid_aq,
        levels=resid_levels,
        linewidths=0.75,
        colors="red",
    )
    plt.clabel(cs, inline=True, fontsize=8)
    ax.set_title("Lower Aquifer")
    label_map_axes(ax, ylabel=False)

**What to look for.** The residuals, simulated minus true head, run from -0.39 to 0.16 m (-1.26 to 0.51 ft); the mean error over all 34 observations is -0.18 m (-0.60 ft) and the root mean square error 0.21 m (0.70 ft). The model is low at all but one observation, most of all at the wells on the western side of the valley, and at most wells the two aquifers are off by about the same amount.

The truth is the same valley simulated on a finer Voronoi grid, with a hydraulic conductivity field for each layer and the clay's own vertical hydraulic conductivity, where this model uses one field for both aquifers. The residuals come from those differences, as they would from the differences between a model and the real aquifer. With field measurements, residuals also carry measurement error and processes the model leaves out, so they are usually larger and less smooth than these.

#### First 10-year transient period

Simulated heads and drawdown at the end of the first transient prediction period.

In [ ]:
totim = totim_end[0]
totim1 = totim_end[1]

In [ ]:
hds = get_heads(totim1)
plot_layers(
    to_display(hds), f"Heads - Transient Period 1, in {length_name()}", levels=levels
)

In [ ]:
ddn = get_heads(totim) - get_heads(totim1)
ddn_levels = MaxNLocator(8).tick_values(0.0, to_display(ddn.max()))
plot_layers(
    to_display(ddn),
    f"Drawdown - Transient Period 1, in {length_name()}",
    levels=ddn_levels,
)

print(f"Maximum drawdown: {to_display(ddn.max()):.2f} {unit(plain=True)}")
v = gwf.output.budget().get_data(text=riv_text, totim=totim1)[0]["q"]
print(f"Induced river infiltration: {np.all(v > 0)}\n{to_display(v, 'flow')}")

**What to look for - drawdown.** "Drawdown" here is the calibration-period head minus the later head, so positive values mark places where the water table has dropped. After ten years of pumping it is largest at the reilly well, 1.49 m (4.89 ft) in layer 5, and 1.39 m (4.57 ft) at the vc well. The water table falls across the whole valley as well, by 0.88 m (2.88 ft) on average over all cells, and least next to the stream, which holds the heads near it.

The induced-infiltration check prints `False`. Every reach exchange stays negative, meaning the aquifer is still discharging to the stream everywhere; nowhere does the stream begin to lose water to the aquifer. The discharge falls, from 56,968 m³/d (2,011,800 ft³/d) at the end of the calibration period to 40,180 m³/d (1,418,900 ft³/d) at the end of this one. The wells are supplied in part by water that would otherwise have reached the stream, which is streamflow depletion without any induced infiltration.

#### Second 10-year transient period

Simulated heads and drawdown at the end of the second transient prediction period.

In [ ]:
totim2 = totim_end[2]

In [ ]:
hds = get_heads(totim2)
plot_layers(
    to_display(hds),
    f"Heads - Transient Period 2 - Prediction, in {length_name()}",
    levels=levels,
)

In [ ]:
ddn = get_heads(totim) - get_heads(totim2)
ddn_levels = MaxNLocator(8).tick_values(0.0, to_display(ddn.max()))
plot_layers(
    to_display(ddn),
    f"Drawdown - Transient Period 2 - Prediction, in {length_name()}",
    levels=ddn_levels,
)

print(f"Maximum drawdown: {to_display(ddn.max()):.2f} {unit(plain=True)}")
v = gwf.output.budget().get_data(text=riv_text, totim=totim2)[0]["q"]
print(f"Induced river infiltration: {np.all(v > 0)}\n{to_display(v, 'flow')}")

**What to look for.** In the second period the prediction well, in layer 5 about 1.1 km (3,500 ft) east of the stream, starts pumping as well. By the end of it the drawdown is 3.11 m (10.21 ft) at the prediction well and 2.76 m (9.04 ft) at the reilly well, and 2.00 m (6.57 ft) on average over the valley. The aquifer's discharge to the stream keeps falling, to 19,966 m³/d (705,100 ft³/d), about a third of what it was at the end of the calibration period, and the reaches beside the prediction well lose the most of theirs: reaches 11 and 12 each give the stream 87 percent less, about 335 m³/d (11,800 ft³/d) instead of about 2,600 m³/d (91,500 ft³/d).

The check still prints `False`, so even after twenty years the stream is gaining along its whole length. Depletion here is the loss of groundwater discharge, not a reversal of the gradient.

### Streamflow results

Streamflow at the gauge near the southern boundary, and the percent reduction in the water the aquifer gives the stream caused by pumping (from the SFR observations when the SFR package is present, otherwise the RIV observations).

In [ ]:
if has_sfr:
    df = gwf.sfr.output.obs().get_dataframe(start_datetime=start_date)
    df["RIV-FLOW"] = to_display(-df["RIV-FLOW"], "streamflow")
else:
    df = gwf.riv.output.obs().get_dataframe(start_datetime=start_date)

# water the aquifer gives the stream (positive)
df["RIV-SWGW"] = to_display(-df["RIV-SWGW"], "streamflow")
df["TOTAL"] = df["RIV-SWGW"]

Q0 = df["TOTAL"].iloc[0]
df["PCT_DIFF"] = -100.0 * (df["TOTAL"] - Q0) / Q0
df

In [ ]:
if has_sfr:
    with flopy.plot.styles.USGSPlot():
        fig, axs = plt.subplots(2, 1, figsize=(9, 3), sharex=True)

        fig.suptitle("Stream gauge near the southern boundary")

        ax = axs[0]
        ax.set_ylim(bottom=0)
        df["RIV-FLOW"].plot(ax=ax, ls="-", marker="o", clip_on=False)
        ax.axhline(0, lw=0.5, color="black")
        ax.set_ylabel(f"Stream flow,\n{unit('streamflow')}")

        ax = axs[1]
        ax.set_ylim(0, 100)
        df["PCT_DIFF"].plot(ax=ax, ls="-", marker="o", clip_on=False)
        ax.axhline(0, lw=0.5, color="black")
        ax.set_ylabel("Reduction in\naquifer discharge\nto the stream, %")
        ax.set_xlabel("Stress Period")

**What to look for.** The top panel is the flow at the gauge, near the southern boundary. The bottom panel is the percent reduction in the water the aquifer gives the stream, relative to the first stress period, which is a different quantity from the one above it: the gauge carries what the whole stream delivers, including the runoff and seepage the mover sends to it, while the exchange is only what the aquifer gives the stream. Both swing from year to year with the rainfall: the reduction is 13 percent in the wet year 2018 and 40 percent in the dry year 2007. Under the swings, pumping takes a growing share; the reduction reaches 65 percent by the end of the simulation, when the gauge carries 0.21 m³/s (7.5 ft³/s), about a third of its flow at the end of the calibration period. (This figure is drawn only when the SFR package is present.)

### Lake level

Simulated lake level through time. The level is written to the same observation file whether or not the LAK package is present (the LAK package writes it when present; the base model's GWF-OBS package writes the head in a high-conductivity lake cell otherwise), so the plot works either way.

In [ ]:
fpth = advanced_ws / f"{name}.lake.obs.csv"
lake_df = flopy.utils.Mf6Obs(
    fpth,
).get_dataframe(start_datetime=start_date)
lake_df["LAKE-STAGE"] = to_display(lake_df["LAKE-STAGE"])
if has_lak:
    # water the lake gains from the aquifer (positive) or loses to it
    lake_df["LAKE-SWGW"] = to_display(lake_df["LAKE-SWGW"], "flow")
lake_df

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(1, 1, figsize=(9, 1.5))

    lake_df["LAKE-STAGE"].plot(ax=ax, ls="-", marker="o", clip_on=False)
    ax.set_ylabel(f"Lake level,\n{length_name()}")
    ax.set_xlabel("Stress Period")

**What to look for.** With the LAK package, this is the simulated lake level; in the base model it is the head in a high-conductivity lake cell, so the plot works either way. The lake starts at 6.50 m (21.33 ft), the level that sets which cells are lake cells, and falls to 4.51 m (14.81 ft) by the end of the simulation, as the pumping lowers the water table around it. The lake leaks to the aquifer in every year but one: in 2018, the wettest year, the water table around the lake rises above it and the aquifer feeds the lake.

**Recap.** Starting from a calibrated base model, the advanced-packages series replaced its simple boundary conditions with advanced packages - `UZF` for recharge, `MAW` for wells, `SFR` for the river, and `LAK` for the lake - and connected them with the `MVR` mover, each built in its own notebook on the shared model in `models/`. This notebook ran that model and evaluated it with calibration targets and across two prediction periods, looking at head residuals against the truth, pumping-induced drawdown, streamflow capture, and the lake level. Because the plots draw only the packages actually present, the same notebook works whether you have built every advanced package or none.